# 01 — Environment Setup & Data Loading

**Pre-condition:** Run `prepare_data.py` locally before this notebook.
It converts BDD100K into filtered YOLO/COCO splits and writes them to
`data_prepared/`. Upload that folder to Google Drive once.

This notebook:
1. Installs dependencies
2. Verifies the GPU environment
3. Mounts Drive and copies the pre-converted splits to `/content/`
4. Validates split image/label counts

In [1]:
import torch

print('Python:', __import__('sys').version)
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')

Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
PyTorch: 2.10.0+cu128
CUDA available: True
GPU: NVIDIA A100-SXM4-40GB
VRAM: 42.4 GB


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [6]:
!pip install -q ultralytics rfdetr fvcore nvidia-ml-py python-dotenv 
!pip install -q /content/drive/MyDrive/FON/master_rad/python_packages/thesis_src-0.3.0-py3-none-any.whl --force-reinstall

In [7]:
from pathlib import Path
import zipfile

DATA_ROOT     = Path('/content/data_prepared')
YOLO_DATA_DIR = DATA_ROOT / 'yolo'

COCO_DATA_DIR = DATA_ROOT / 'coco'
CONFIGS_DIR   = DATA_ROOT / 'configs'

ZIP_SRC = Path('/content/drive/MyDrive/FON/master_rad/bdd100k_data/bdd100k_data.zip')

if YOLO_DATA_DIR.exists():
    print('Already extracted — skipping.')
else:
    print(f'Extracting from Drive ({ZIP_SRC.stat().st_size / 1e9:.1f} GB)...')
    with zipfile.ZipFile(ZIP_SRC) as z:
        z.extractall('/content/')
    print('Done.')

Already extracted — skipping.


In [8]:
import json

conditions = json.loads((CONFIGS_DIR / 'conditions.json').read_text())
splits = ['clear_day/train', 'clear_day/val'] + conditions['adverse']

print(f'{"Split":<30} {"Images":>8} {"Labels":>8} {"OK":>4}')
print('-' * 55)
for split in splits:
    img_dir = YOLO_DATA_DIR / split / 'images'
    lbl_dir = YOLO_DATA_DIR / split / 'labels'
    n_img = len(list(img_dir.glob('*.jpg'))) if img_dir.exists() else 0
    n_lbl = len(list(lbl_dir.glob('*.txt'))) if lbl_dir.exists() else 0
    ok = '✓' if n_img > 0 and n_img == n_lbl else '✗'
    print(f'{split:<30} {n_img:>8} {n_lbl:>8} {ok:>4}')

Split                            Images   Labels   OK
-------------------------------------------------------
clear_day/train                   12454    12454    ✓
clear_day/val                      1764     1764    ✓
clear_dawn_dusk                    2311     2311    ✓
clear_night                        3274     3274    ✓
overcast_dawn_dusk                 1329     1329    ✓
overcast_daytime                   8590     8590    ✓
partly_cloudy_dawn_dusk             665      665    ✓
partly_cloudy_daytime              4900     4900    ✓
rainy_dawn_dusk                     383      383    ✓
rainy_daytime                      2918     2918    ✓
rainy_night                        2494     2494    ✓
snowy_dawn_dusk                     510      510    ✓
snowy_daytime                      3284     3284    ✓
snowy_night                        2522     2522    ✓
